# EMS Readiness Optimization - Complete Pipeline

This all-in-one notebook runs the entire EMS optimization analysis from data loading through statistical analysis and visualization.

**WARNING: Full execution requires Colab Pro (4-8+ hours runtime, high memory)**

Sections:
1. Setup and Data Loading
2. Exploratory Data Analysis
3. Demand Modeling (NHPP)
4. Service Modeling (Travel Time, Service Time)
5. Optimization (P0, P1, P2)
6. Discrete-Event Simulation
7. Statistical Analysis
8. Visualization and Reporting

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
DOWNLOAD_OUTPUTS = False  # Set True to download output files
SAVE_TO_DRIVE = False     # Set True to save outputs to Google Drive

if IN_COLAB:
    print("Running in Google Colab - installing dependencies...")
    !pip install -q simpy pulp pyyaml tqdm geopandas folium
    if not os.path.exists('ems-optimization'):
        !git clone --depth=1 https://github.com/cnsp/ems-optimization.git
    PROJECT_ROOT = '/content/ems-optimization'
else:
    print("Running locally - detecting project root...")
    # Robust local path detection: walk up from notebook location to find project root
    # Works whether launched from notebook dir, project root, or anywhere else
    _nb_dir = os.getcwd()
    _candidate = _nb_dir
    PROJECT_ROOT = None
    for _ in range(10):  # Walk up at most 10 levels
        if os.path.isfile(os.path.join(_candidate, 'requirements.txt')) and \
           os.path.isdir(os.path.join(_candidate, 'src', 'ems_readiness')):
            PROJECT_ROOT = _candidate
            break
        _parent = os.path.dirname(_candidate)
        if _parent == _candidate:
            break
        _candidate = _parent
    if PROJECT_ROOT is None:
        # Fallback: assume notebook is at notebooks/colab_standalone/individual/
        PROJECT_ROOT = os.path.abspath(os.path.join(_nb_dir, '..', '..', '..'))
    print(f"  Project root: {PROJECT_ROOT}")

sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
CONFIGS_DIR = os.path.join(PROJECT_ROOT, 'configs')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Optional Google Drive save
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/EMS_Optimization_Results'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Saving outputs to: {DRIVE_DIR}")

def save_output(fig_or_df, filename, subdir=''):
    """Helper to save outputs with optional download/drive save."""
    out_dir = os.path.join(RESULTS_DIR, subdir) if subdir else RESULTS_DIR
    os.makedirs(out_dir, exist_ok=True)
    filepath = os.path.join(out_dir, filename)
    if isinstance(fig_or_df, pd.DataFrame):
        fig_or_df.to_csv(filepath, index=True)
    elif hasattr(fig_or_df, 'savefig'):
        fig_or_df.savefig(filepath, bbox_inches='tight', dpi=150)
    if IN_COLAB and DOWNLOAD_OUTPUTS:
        from google.colab import files
        files.download(filepath)
    if IN_COLAB and SAVE_TO_DRIVE:
        import shutil
        drive_path = os.path.join(DRIVE_DIR, subdir)
        os.makedirs(drive_path, exist_ok=True)
        shutil.copy(filepath, os.path.join(drive_path, filename))

print("Setup complete. PROJECT_ROOT:", PROJECT_ROOT)

---
# Phase 1: Data Loading and Verification

In [ ]:
import glob, yaml

print("=== VERIFYING DATA FILES ===")
raw_files = glob.glob(os.path.join(RAW_DIR, '*'))
print(f"Raw data files: {len(raw_files)}")
for f in sorted(raw_files):
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB)")

processed_files = glob.glob(os.path.join(PROCESSED_DIR, '*'))
print(f"\nProcessed data files: {len(processed_files)}")
for f in sorted(processed_files):
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB)")

# Load key datasets
crashes = pd.read_csv(os.path.join(PROCESSED_DIR, 'crashes_manhattan.csv'),
                     parse_dates=['crash_datetime'])
firehouses = pd.read_csv(os.path.join(PROCESSED_DIR, 'firehouses_manhattan.csv'))
dm = pd.read_csv(os.path.join(PROCESSED_DIR, 'distance_matrix_firehouse_precinct.csv'), index_col=0)
dm.columns = dm.columns.astype(str)
precinct_demand = pd.read_csv(os.path.join(PROCESSED_DIR, 'demand_lambda_precinct.csv'))
hourly_lambda = pd.read_csv(os.path.join(PROCESSED_DIR, 'demand_lambda_hourly.csv'))
dow_lambda = pd.read_csv(os.path.join(PROCESSED_DIR, 'demand_lambda_dow.csv'))

print(f"\nCrashes: {len(crashes):,}")
print(f"Firehouses: {len(firehouses)}")
print(f"Distance matrix: {dm.shape}")
print(f"Precincts: {len(precinct_demand)}")

---
# Phase 2: Exploratory Data Analysis

In [ ]:
crashes['hour'] = crashes['crash_datetime'].dt.hour
crashes['dow'] = crashes['crash_datetime'].dt.dayofweek
crashes['day_name'] = crashes['crash_datetime'].dt.day_name()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Hourly
hourly_counts = crashes.groupby('hour').size()
hourly_counts.plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Crashes by Hour of Day')
axes[0,0].set_xlabel('Hour')

# Day of week
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_counts = crashes.groupby('day_name').size().reindex(dow_order)
dow_counts.plot(kind='bar', ax=axes[0,1], color='darkorange')
axes[0,1].set_title('Crashes by Day of Week')
axes[0,1].tick_params(axis='x', rotation=45)

# CBD vs Non-CBD
if 'in_cbd' in crashes.columns:
    cbd_hourly = crashes[crashes['in_cbd']==True].groupby('hour').size()
    non_cbd_hourly = crashes[crashes['in_cbd']==False].groupby('hour').size()
    axes[1,0].plot(cbd_hourly.index, cbd_hourly.values, 'b-o', label='CBD', markersize=3)
    axes[1,0].plot(non_cbd_hourly.index, non_cbd_hourly.values, 'r-s', label='Non-CBD', markersize=3)
    axes[1,0].legend()
    axes[1,0].set_title('Hourly: CBD vs Non-CBD')
    axes[1,0].set_xlabel('Hour')

# Monthly trend
crashes['year_month'] = crashes['crash_datetime'].dt.to_period('M')
monthly = crashes.groupby('year_month').size()
monthly.plot(ax=axes[1,1], color='seagreen')
axes[1,1].set_title('Monthly Crash Volume')

plt.tight_layout()
save_output(fig, 'eda_overview.png', 'figures/eda')
plt.show()
print(f"Total crashes: {len(crashes):,}")
print(f"CBD: {crashes['in_cbd'].sum():,} ({100*crashes['in_cbd'].mean():.1f}%)")

---
# Phase 3: Demand Modeling (NHPP)

In [ ]:
from ems_readiness.demand.arrival_generator import NHPPArrivalGenerator

gen = NHPPArrivalGenerator.from_tables(data_dir=PROCESSED_DIR)
print(f"Base rate: {gen.base_rate:.4f} crashes/hour")

# Generate sample week
week_arrivals = []
for day in range(7):
    day_arr = gen.generate_arrivals(n_hours=24, start_hour=0, dow=day, rng=42+day)
    day_arr['day'] = day
    week_arrivals.append(day_arr)
all_arrivals = pd.concat(week_arrivals, ignore_index=True)
print(f"Simulated week arrivals: {len(all_arrivals):,}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(hourly_lambda['hour'], hourly_lambda['lambda_per_hour'], 'b-o', markersize=4)
axes[0].set_title('Hourly Lambda')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Lambda (crashes/hr)')

axes[1].bar(dow_lambda['day_name'], dow_lambda['factor'], color='darkorange')
axes[1].set_title('DOW Factor')
axes[1].tick_params(axis='x', rotation=45)

prec_sorted = precinct_demand.sort_values('lambda_per_hour', ascending=True)
axes[2].barh(prec_sorted['precinct'].astype(str), prec_sorted['lambda_per_hour'], color='seagreen')
axes[2].set_title('Precinct Rates')
plt.tight_layout()
save_output(fig, 'demand_profiles.png', 'figures/demand')
plt.show()

---
# Phase 4: Service Modeling

In [ ]:
from ems_readiness.service.travel_time import build_travel_time_matrix
from ems_readiness.service.service_time import ServiceTimeModel

tt = build_travel_time_matrix(dm, speed_mph=20.0, hour_of_day=None)
print(f"Travel time matrix: mean={tt.values.mean():.2f} min, max={tt.values.max():.2f} min")

stm = ServiceTimeModel(mean_minutes=25.0, std_minutes=10.0, distribution='lognormal')
samples = stm.sample(size=10000, rng=42)
print(f"Service time: mean={np.mean(samples):.2f}, median={np.median(samples):.2f}, p90={np.percentile(samples,90):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(tt.iloc[:10, :10], annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Travel Time Matrix (subset, minutes)')
axes[1].hist(samples, bins=50, density=True, color='steelblue', edgecolor='white')
axes[1].set_title(f'Service Time Distribution (mean={np.mean(samples):.1f} min)')
axes[1].set_xlabel('Minutes')
plt.tight_layout()
save_output(fig, 'service_overview.png', 'figures/service')
plt.show()

---
# Phase 5: Optimization

In [ ]:
from ems_readiness.optimization.policies import uniform_allocation, demand_proportional_allocation
from ems_readiness.optimization.models import build_demand_weighted, extract_allocation
import pulp

demand = precinct_demand.set_index('precinct')['lambda_per_hour']
demand.index = demand.index.astype(str)
capacity = 2

K_VALUES = [15, 20, 25, 30, 35, 40]
allocations = {}

for K in K_VALUES:
    allocations[('P0', K)] = uniform_allocation(dm.index.tolist(), K=K, capacity=capacity)
    allocations[('P1', K)] = demand_proportional_allocation(tt, demand, K=K, capacity=capacity)
    prob = build_demand_weighted(tt, demand, K=K, capacity=capacity)
    prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=120))
    allocations[('P2', K)] = extract_allocation(prob)
    print(f"K={K}: P0={allocations[('P0',K)].sum():.0f}, P1={allocations[('P1',K)].sum():.0f}, P2={allocations[('P2',K)].sum():.0f}")

# Save allocations
alloc_dir = os.path.join(RESULTS_DIR, 'optimization')
os.makedirs(alloc_dir, exist_ok=True)
for K in K_VALUES:
    adf = pd.DataFrame({p: allocations[(p, K)] for p in ['P0', 'P1', 'P2']})
    adf.to_csv(os.path.join(alloc_dir, f'allocations_K{K}.csv'))
print("Allocations saved.")

---
# Phase 6: Discrete-Event Simulation

**Set RUN_MODE below.** Use `'demo'` for quick test or `'full'` for production.

In [ ]:
import time as time_module
from ems_readiness.simulation.engine import EMSSimulation

RUN_MODE = 'demo'  # Change to 'full' for production

if RUN_MODE == 'demo':
    SIM_K_VALUES = [20]
    SIM_POLICIES = ['P0', 'P1', 'P2']
    NUM_REPS = 5
else:
    SIM_K_VALUES = K_VALUES
    SIM_POLICIES = ['P0', 'P1', 'P2']
    NUM_REPS = 30

with open(os.path.join(CONFIGS_DIR, 'simulation.yaml')) as f:
    sim_config = yaml.safe_load(f)
HORIZON = 168
SEED_BASE = 42

all_sim_results = []
t0 = time_module.time()
total = len(SIM_K_VALUES) * len(SIM_POLICIES) * NUM_REPS
done = 0

for K in SIM_K_VALUES:
    for policy in SIM_POLICIES:
        key = (policy, K)
        if key not in allocations:
            continue
        alloc = allocations[key]
        for rep in range(NUM_REPS):
            seed = SEED_BASE + rep
            sim = EMSSimulation(policy_allocation=alloc, config=sim_config, seed=seed,
                              data_dir='data/processed', project_root=PROJECT_ROOT)
            sim.run(horizon_hours=HORIZON)
            s = sim.get_results()['summary']
            s['policy'], s['K'], s['replication'], s['seed'] = policy, K, rep, seed
            all_sim_results.append(s)
            done += 1
        elapsed = time_module.time() - t0
        print(f"{policy} K={K}: {NUM_REPS} reps done | {done}/{total} ({100*done/total:.0f}%) | {elapsed:.0f}s")

results_df = pd.DataFrame(all_sim_results)
sim_dir = os.path.join(RESULTS_DIR, 'simulation')
os.makedirs(sim_dir, exist_ok=True)
results_df.to_csv(os.path.join(sim_dir, 'simulation_results_all.csv'), index=False)
print(f"\nDone! {len(results_df)} results saved. Total time: {time_module.time()-t0:.0f}s")

---
# Phase 7: Statistical Analysis

In [ ]:
from scipy import stats

print("=== STATISTICAL ANALYSIS ===\n")

K_test = results_df['K'].mode().values[0]
groups = {}
for policy in sorted(results_df['policy'].unique()):
    data = results_df[(results_df['policy']==policy) & (results_df['K']==K_test)]['response_time_mean'].values
    if len(data) > 0:
        groups[policy] = data
        print(f"{policy}: mean={np.mean(data):.4f}, std={np.std(data,ddof=1):.4f}, n={len(data)}")

if len(groups) >= 2:
    F, p = stats.f_oneway(*groups.values())
    print(f"\nOne-Way ANOVA: F={F:.4f}, p={p:.2e}")

    # Pairwise
    from itertools import combinations
    for a, b in combinations(groups.keys(), 2):
        t, p_val = stats.ttest_ind(groups[a], groups[b], equal_var=False)
        pooled = np.sqrt((np.std(groups[a],ddof=1)**2 + np.std(groups[b],ddof=1)**2)/2)
        d = abs(np.mean(groups[a]) - np.mean(groups[b])) / pooled if pooled > 0 else 0
        print(f"  {a} vs {b}: diff={np.mean(groups[a])-np.mean(groups[b]):.4f}, t={t:.4f}, p={p_val:.2e}, Cohen's d={d:.4f}")

---
# Phase 8: Visualization and Reporting

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'P0': '#4878CF', 'P1': '#E8A838', 'P2': '#6ACC65'}
labels = {'P0': 'P0 (Baseline)', 'P1': 'P1 (Demand-Prop)', 'P2': 'P2 (Optimized)'}

for policy in sorted(results_df['policy'].unique()):
    grp = results_df[results_df['policy']==policy].groupby('K')
    m = grp['response_time_mean'].mean()
    s = grp['response_time_mean'].std()
    axes[0].errorbar(m.index, m.values, yerr=1.96*s/np.sqrt(NUM_REPS), fmt='-o',
                    label=labels.get(policy,policy), color=colors.get(policy), capsize=4, linewidth=2)
    m2 = grp['coverage_fraction'].mean()
    s2 = grp['coverage_fraction'].std()
    axes[1].errorbar(m2.index, m2.values*100, yerr=1.96*s2*100/np.sqrt(NUM_REPS), fmt='-s',
                    label=labels.get(policy,policy), color=colors.get(policy), capsize=4, linewidth=2)
    m3 = grp['queue_fraction'].mean()
    axes[2].plot(m3.index, m3.values*100, '-^', label=labels.get(policy,policy),
                color=colors.get(policy), linewidth=2)

for i, (ylabel, title) in enumerate([('Mean Response Time (min)', 'Response Time'),
                                      ('Coverage (%)', 'Coverage (8 min)'),
                                      ('Queue Fraction (%)', 'Queueing Rate')]):
    axes[i].set_xlabel('Fleet Size (K)')
    axes[i].set_ylabel(ylabel)
    axes[i].set_title(title)
    axes[i].legend(fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
save_output(fig, 'main_results.png', 'figures/publication')
plt.show()

print("\n=== PIPELINE COMPLETE ===")
print(f"All results saved to: {RESULTS_DIR}")